In [5]:
!pip install dash

   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/7.2 MB 176.5 kB/s eta 0:00:39
   -- ------------------------------------- 0.5/7.2 MB 176.5 kB/s eta 0:00:39
   -- ----------

In [22]:
pip install dash pandas plotly werkzeug

In [25]:
import dash
from dash import dcc, html, Input, Output, State
import plotly.express as px
import pandas as pd
import numpy as np

# For password hashing
from werkzeug.security import generate_password_hash, check_password_hash


# -------------------------
# Secure User Database
# -------------------------

USER_DB = {

    "admin": generate_password_hash("1234"),

    "mustafeez": generate_password_hash("pass123")

}


# -------------------------
# Generate Stock Data
# -------------------------

np.random.seed(42)

dates = pd.date_range("2024-01-01", "2025-12-31")

companies = ["TCS", "Infosys", "Wipro"]

data = []

for company in companies:

    price = 1000 + np.random.randint(0, 200)

    for date in dates:

        price += np.random.randint(-5, 8)

        volume = np.random.randint(1000, 8000)

        data.append({

            "Date": date,
            "Company": company,
            "Price": round(price, 2),
            "Volume": volume

        })

df = pd.DataFrame(data)


# -------------------------
# App
# -------------------------

app = dash.Dash(__name__)

app.title = "Secure Stock Dashboard"


# -------------------------
# Layout
# -------------------------

app.layout = html.Div([

    dcc.Location(id="url"),

    dcc.Store(id="session", data=False),

    html.Div(id="page")

])


# -------------------------
# Login Page
# -------------------------

login_page = html.Div([

    html.H1("Secure Login", style={'textAlign': 'center'}),

    dcc.Input(

        id="username",

        placeholder="Enter Username",

        type="text"

    ),

    html.Br(),

    dcc.Input(

        id="password",

        placeholder="Enter Password",

        type="password"

    ),

    html.Br(),

    html.Button("Login", id="login-btn"),

    html.Div(id="login-msg")

], style={'textAlign': 'center', 'marginTop': '100px'})


# -------------------------
# Dashboard Page
# -------------------------

dashboard_page = html.Div([

    html.H1("Stock Dashboard"),

    html.Button("Logout", id="logout-btn"),

    dcc.Dropdown(

        id="company",

        options=[{"label": c, "value": c} for c in df["Company"].unique()],

        value="TCS"

    ),

    dcc.Graph(id="chart")

])


# -------------------------
# Login Authentication
# -------------------------

@app.callback(

    Output("session", "data"),

    Output("login-msg", "children"),

    Input("login-btn", "n_clicks"),

    State("username", "value"),

    State("password", "value"),

    prevent_initial_call=True

)

def login(n, username, password):

    if username in USER_DB:

        if check_password_hash(USER_DB[username], password):

            return True, "Login Successful"

    return False, "Invalid Credentials"


# -------------------------
# Logout
# -------------------------

@app.callback(

    Output("session", "data", allow_duplicate=True),

    Input("logout-btn", "n_clicks"),

    prevent_initial_call=True

)

def logout(n):

    return False


# -------------------------
# Page Control
# -------------------------

@app.callback(

    Output("page", "children"),

    Input("session", "data")

)

def display_page(session):

    if session:

        return dashboard_page

    return login_page


# -------------------------
# Chart
# -------------------------

@app.callback(

    Output("chart", "figure"),

    Input("company", "value")

)

def update_chart(company):

    filtered = df[df["Company"] == company]

    fig = px.line(

        filtered,

        x="Date",

        y="Price",

        template="plotly_dark",

        color_discrete_sequence=["cyan"]

    )

    return fig


# -------------------------
# Run
# -------------------------

app.run(debug=True)